# CYGNSS L1: observation quality and implied observation error

Two related questions, kept apart because they need different measures and
different runs:

1. **How is the analysis performing, and where?** — needs the assimilating run.
   Uses `improved`, `right_direction`, `gain_proxy`.
2. **How large is the observation error, and what predicts it?** — best answered
   on the **open loop**, where no analysis has fed back on its own innovations.
   Uses implied observation error from the innovation identity.

| run | rows | period | role |
|---|---:|---|---|
| `DAv8_M36_AZ_paired_cygl1_dense_gated` | 202,610 | Jan 2020 – Mar 2021 | analysis response |
| `OLv8_M36_all_sensors_AZ_scaled` | 469,099 | Jan 2020 – Dec 2022 | observation error |

Arizona M36 domain, species `CYGNSS_L1_DDM3X5_CROP_SCALAR`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from IPython.display import display

plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True,
                     'grid.color': '0.92', 'axes.axisbelow': True})


def find_project(name='CYGNSS_L1_AZ'):
    """Locate the project dir regardless of the kernel's working directory."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        if base.name == name:
            return base
        cand = base / 'projects' / name
        if cand.is_dir():
            return cand
    raise FileNotFoundError(f'could not locate {name} from {Path.cwd()}')


PROJECT = find_project()
PER_OBS = PROJECT / 'output' / 'cygnss_da_quality_bundle' / 'per_obs'
EXTENT = (-118.6, -105.2, 28.6, 40.4)   # AZ domain

COLS = ['sp_lon', 'sp_lat', 'cycle_utc', 'tilenum',
        'obs', 'fcst', 'ana', 'obsvar', 'fcstvar',
        'd_f', 'd_a', 'delta_h', 'K', 'gain_proxy', 'gain_proxy_valid',
        'improved', 'right_direction',
        'coherency_ratio', 'ddm_snr', 'brcs_crop_sum', 'quality_score',
        'sp_rx_gain', 'sp_inc_angle', 'srtm_slope',
        'coefficient_weighted_opacity', 'modis_land_cover', 'MWRTM_VEGCLS']


def load(fname):
    p = PER_OBS / fname
    if not p.exists():
        raise FileNotFoundError(f'not found: {p}')
    df = pd.read_csv(p, usecols=COLS)
    for c in ['gain_proxy_valid', 'improved', 'right_direction']:
        df[c] = df[c].astype(str).str.lower().eq('true')
    return df


da = load('cygnss_da_performance_with_quality_all.csv')   # assimilating
ol = load('cygnss_ol_innovations_with_quality_all.csv')   # open loop
print(f'DA {len(da):,} obs   {da.cycle_utc.min()} .. {da.cycle_utc.max()}')
print(f'OL {len(ol):,} obs   {ol.cycle_utc.min()} .. {ol.cycle_utc.max()}')

## 1. What is in the two tables

Identical 51-column schema. The columns used here:

| column | meaning |
|---|---|
| `d_f` | `obs − fcst`, the **innovation** (O−F), in dB |
| `d_a` | `obs − ana`, the analysis residual (O−A) |
| `delta_h` | `ana − fcst`, the increment applied, in observation space |
| `obsvar` | prescribed observation error variance, **after scaling** |
| `fcstvar` | ensemble forecast error variance in obs space, i.e. HPHᵀ |
| `K` | `fcstvar / (fcstvar + obsvar)` — scalar Kalman gain |

In the open loop there is no analysis, so `delta_h` is identically zero and
`d_a == d_f`. Only `d_f` carries information there — which is exactly what the
observation-error analysis needs.

In [ ]:
chk = pd.DataFrame({
    'DA': [len(da), da.tilenum.nunique(), da.cycle_utc.nunique(),
           (da.delta_h == 0).mean(), da.groupby(['tilenum', 'cycle_utc']).size().max()],
    'OL': [len(ol), ol.tilenum.nunique(), ol.cycle_utc.nunique(),
           (ol.delta_h == 0).mean(), ol.groupby(['tilenum', 'cycle_utc']).size().max()],
}, index=['observations', 'tiles', 'cycles',
          'fraction with zero increment', 'max obs per tile-cycle'])
display(chk)
print('max obs per tile-cycle = 1 in both -> no pseudo-replication in binned statistics')

## 2. Is the analysis responding sensibly?  *(assimilating run)*

A correctly functioning filter must pull *harder* where the gain is larger, so
`improved` and `gain_proxy` should both rise with `K`. That rise is a **positive
control**.

In [ ]:
valid = da[da.gain_proxy_valid]
kq = pd.qcut(valid.K, 5, labels=False, duplicates='drop')
resp = valid.groupby(kq).agg(
    K_lo=('K', 'min'), K_hi=('K', 'max'), n=('K', 'size'),
    improved=('improved', 'mean'),
    right_direction=('right_direction', 'mean'),
    gain_proxy=('gain_proxy', 'median'))
display(resp.round(3))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
x = np.arange(len(resp))
ax[0].bar(x - 0.2, resp.improved, 0.4, label='improved', color='#4c78a8')
ax[0].bar(x + 0.2, resp.right_direction, 0.4, label='right direction', color='#72b7b2')
ax[0].set_ylim(0.5, 0.85); ax[0].legend(frameon=False); ax[0].set_ylabel('fraction')
ax[0].set_title('(a) response rises with gain', loc='left', fontweight='bold')
ax[1].bar(x, resp.gain_proxy, 0.55, color='#e45756')
ax[1].set_ylabel('median gain_proxy')
ax[1].set_title('(b) and scales with it', loc='left', fontweight='bold')
for a in ax:
    a.set_xticks(x)
    a.set_xticklabels([f'{lo:.2f}-{hi:.2f}' for lo, hi in zip(resp.K_lo, resp.K_hi)], fontsize=8)
    a.set_xlabel('Kalman gain K, quintile')
fig.tight_layout()

**Read:** the filter behaves as the Kalman equations require — 73% of updates move
the right way, and the response scales with the gain.

Keep that scaling in mind when reading stratifications by quality field. `improved`
is more often true wherever `K` is large, and `K` is a property of the *ensemble* as
much as the observation. Good for describing where the analysis performs well; not a
sufficient basis on its own for choosing a screening threshold — which is why
everything from §4 onward uses the open loop instead.

## 3. Where the observations are  *(open loop, 3 years)*

In [ ]:
def binned(df, value, res=0.5, how='mean', minc=20):
    """Bin a per-observation quantity onto a regular lon/lat grid."""
    lon_edges = np.arange(EXTENT[0], EXTENT[1] + res, res)
    lat_edges = np.arange(EXTENT[2], EXTENT[3] + res, res)
    ix = np.digitize(df.sp_lon, lon_edges) - 1
    iy = np.digitize(df.sp_lat, lat_edges) - 1
    ok = (ix >= 0) & (ix < len(lon_edges) - 1) & (iy >= 0) & (iy < len(lat_edges) - 1)
    grid = np.full((len(lat_edges) - 1, len(lon_edges) - 1), np.nan)
    cnt = np.zeros_like(grid)
    s = pd.DataFrame({'ix': ix[ok], 'iy': iy[ok], 'v': np.asarray(value)[ok]})
    g = s.groupby(['iy', 'ix']).v
    agg = g.mean() if how == 'mean' else g.size()
    n = g.size()
    for (j, i), v in agg.items():
        cnt[j, i] = n.loc[(j, i)]
        grid[j, i] = v
    grid[cnt < minc] = np.nan
    return lon_edges, lat_edges, np.ma.masked_invalid(grid)


def basemap(ax):
    ax.set_extent(EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='0.94', zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale('50m'), facecolor='white', zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale('50m'), lw=0.5, edgecolor='0.3', zorder=3)
    ax.add_feature(cfeature.BORDERS.with_scale('50m'), lw=0.4, edgecolor='0.4', zorder=3)
    ax.add_feature(cfeature.NaturalEarthFeature(
        'cultural', 'admin_1_states_provinces_lakes', '50m', facecolor='none'),
        lw=0.4, edgecolor='0.4', zorder=3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.9),
                         subplot_kw={'projection': ccrs.PlateCarree()})

panels = [(np.ones(len(ol)), 'count', 'viridis', '(a) observation count', None),
          (ol.d_f, 'mean', 'RdBu_r', '(b) mean innovation O$-$F (dB)', 'sym'),
          (ol.coherency_ratio, 'mean', 'magma', '(c) mean coherency ratio', None)]
for ax, (v, how, cmap, title, norm) in zip(axes, panels):
    lo, la, gr = binned(ol, v, how=how, minc=1 if how == 'count' else 20)
    basemap(ax)
    kw = {}
    if norm == 'sym':
        lim = np.nanpercentile(np.abs(gr.compressed()), 98)
        kw = {'vmin': -lim, 'vmax': lim}
    m = ax.pcolormesh(lo, la, gr, cmap=cmap, transform=ccrs.PlateCarree(), zorder=1, **kw)
    fig.colorbar(m, ax=ax, fraction=0.045)
    ax.set_title(title, loc='left', fontweight='bold', fontsize=10.5)
fig.tight_layout()

## 4. Observation error from the innovation identity

For an unbiased forecast, with observation errors uncorrelated with forecast errors:

$$\mathrm{Var}(O-F) = R + HPH^{T}$$

`fcstvar` **is** $HPH^{T}$ — GEOSldas writes the per-observation ensemble variance of
$H(x)$ straight into the ObsFcstAna file. So:

$$\hat{R} = \mathrm{Var}(d_f) - \overline{\texttt{fcstvar}}$$

Bin observations by any candidate quality field. If $\hat{R}$ changes across bins
while `fcstvar` does not, the difference is **observation error** — and unlike the
response metrics, it owes nothing to the gain.

Using the **open loop** removes the last concern: with no analysis, the forecast
cannot have been shaped by previous increments from these same observations.

$\hat{R}$ absorbs instrument noise, *observation-operator* error and representativeness
error together. It is 'everything the filter cannot explain', not a pure instrument
property. It is also computed from `Var()`, so it is **blind to bias** — see §6.

In [ ]:
def implied_R(df, field=None, bins=5, minn=2000, categorical=False):
    """Implied observation error by bin of `field` (or overall if None)."""
    d = df.dropna(subset=['d_f', 'fcstvar'] + ([field] if field else []))
    if field is None:
        key = pd.Series(0, index=d.index)
    elif categorical:
        key = d[field]
    else:
        key = pd.qcut(d[field], bins, labels=False, duplicates='drop')
    g = d.groupby(key).agg(n=('d_f', 'size'),
                           lo=(field or 'd_f', 'min'), hi=(field or 'd_f', 'max'),
                           bias=('d_f', 'mean'), var=('d_f', 'var'),
                           fcstvar=('fcstvar', 'mean'), obsvar=('obsvar', 'mean'),
                           K=('K', 'mean'))
    g = g[g.n >= minn]
    g['R'] = g['var'] - g.fcstvar
    g['errstd'] = np.sqrt(g.R.clip(lower=0))
    return g


both = pd.concat([implied_R(ol).assign(run='OL'), implied_R(da).assign(run='DA')])
display(both.set_index('run')[['n', 'bias', 'var', 'fcstvar', 'R', 'errstd']].round(3))
print('configured errstd in these runs: 4.4 dB (DA sweep) / 3.0 dB (paired family)')

## 5. Which quality fields carry information?  *(open loop)*

For each candidate, the spread in implied $\hat{R}$ across its bins — and alongside it
the spread in `fcstvar`, the check that we are not re-measuring the gain. A useful
field has **large $\Delta R$ and small $\Delta$`fcstvar`**.

In [ ]:
FIELDS = ['coherency_ratio', 'ddm_snr', 'quality_score', 'coefficient_weighted_opacity',
          'brcs_crop_sum', 'sp_rx_gain', 'sp_inc_angle', 'srtm_slope']

rank = []
for f in FIELDS:
    g = implied_R(ol, f)
    if len(g) < 3:
        continue
    gd = implied_R(da, f)
    rank.append({'field': f,
                 'dR_OL': g.R.max() - g.R.min(),
                 'ratio_OL': g.R.max() / max(g.R.min(), 1e-9),
                 'd_fcstvar_OL': g.fcstvar.max() - g.fcstvar.min(),
                 'dR_DA': gd.R.max() - gd.R.min() if len(gd) >= 3 else np.nan})
rank = pd.DataFrame(rank).sort_values('dR_OL', ascending=False).reset_index(drop=True)
display(rank.round(3))

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.8), sharey=True)
y = np.arange(len(rank))[::-1]
ax[0].barh(y + 0.18, rank.dR_OL, 0.36, color='#4c78a8', label='open loop')
ax[0].barh(y - 0.18, rank.dR_DA, 0.36, color='#9ecae9', label='assimilating')
ax[0].set_xlabel(r'$\Delta \hat{R}$  (dB$^2$)  — signal'); ax[0].legend(frameon=False)
ax[0].set_title('(a) how much observation error varies', loc='left', fontweight='bold')
ax[1].barh(y, rank.d_fcstvar_OL, 0.55, color='#b279a2')
ax[1].set_xlabel(r'$\Delta$ fcstvar  (dB$^2$)  — confound, want small')
ax[1].set_title('(b) does the gain vary too?', loc='left', fontweight='bold')
ax[0].set_yticks(y); ax[0].set_yticklabels(rank.field)
fig.tight_layout()

## 6. Coherency ratio in detail

Best field on both criteria, in both runs. Note the **bias** column — coherency
predicts a systematic offset, not just scatter, and $\hat{R}$ cannot see it.

In [ ]:
coh_ol = implied_R(ol, 'coherency_ratio')
coh_da = implied_R(da, 'coherency_ratio')
display(coh_ol[['n', 'lo', 'hi', 'bias', 'var', 'fcstvar', 'obsvar', 'R', 'errstd']].round(3))

fig, ax = plt.subplots(1, 3, figsize=(13.5, 3.6))
x = np.arange(len(coh_ol))
lab = [f'{a:.2f}-{b:.2f}' for a, b in zip(coh_ol.lo, coh_ol.hi)]
ax[0].bar(x - 0.19, coh_ol.errstd, 0.38, color='#4c78a8', label='open loop')
ax[0].bar(x + 0.19, coh_da.errstd.reindex(coh_ol.index), 0.38, color='#9ecae9', label='assimilating')
ax[0].axhline(4.4, color='#e45756', ls='--', lw=1.2)
ax[0].text(0.05, 4.5, 'configured 4.4 dB', color='#e45756', fontsize=8)
ax[0].set_ylabel('implied errstd (dB)'); ax[0].legend(frameon=False, fontsize=8)
ax[0].set_title('(a) observation error — both runs agree', loc='left', fontweight='bold')
ax[1].bar(x, coh_ol.bias, 0.6, color=['#e45756' if v < 0 else '#54a24b' for v in coh_ol.bias])
ax[1].axhline(0, color='0.3', lw=0.9); ax[1].set_ylabel('mean O$-$F (dB)')
ax[1].set_title('(b) bias — opposite signs at the ends', loc='left', fontweight='bold')
ax[2].bar(x - 0.19, coh_ol.fcstvar, 0.38, color='#b279a2', label='fcstvar (HPH$^T$)')
ax[2].bar(x + 0.19, coh_ol.R, 0.38, color='#f58518', label=r'implied $\hat{R}$')
ax[2].set_ylabel('dB$^2$'); ax[2].legend(frameon=False, fontsize=8)
ax[2].set_title('(c) confound flat, signal is not', loc='left', fontweight='bold')
for a in ax:
    a.set_xticks(x); a.set_xticklabels(lab, fontsize=8, rotation=20)
    a.set_xlabel('coherency ratio')
fig.tight_layout()

**Read:**

- The bottom quintile carries about **3× the observation error** of the middle bins,
  and the two runs agree — so this is not the analysis feeding back on its own
  innovations. That was the acceptance check for using this as a screen.
- Above ~0.5 the curve is flat, so this is a **threshold, not a ranking**.
- The configured 4.4 dB matches the *worst* quintile and is too large for the other 80%.
- The **bias swings monotonically** from about −1 dB to +1 dB. Low-coherency
  observations say the surface is drier than the model; high-coherency say wetter.
  `Var()` removes the mean, so $\hat{R}$ is blind to this and it needs handling
  separately from any error-variance decision.

### The screen: one-sided, not two-sided

Dropping the *top* quintile as well costs 20% of the data for little error reduction
and makes the retained population **more** biased, because it removes the
positive-bias group that was offsetting the negative one.

In [ ]:
P20, P80 = np.percentile(ol.coherency_ratio.dropna(), [20, 80])
print(f'P20 = {P20:.3f}   P80 = {P80:.3f}')

opts = {'all observations': ol.coherency_ratio.notna(),
        'one-sided  >= P20': ol.coherency_ratio >= P20,
        'two-sided  P20-P80': (ol.coherency_ratio >= P20) & (ol.coherency_ratio <= P80)}
rows = []
for name, m in opts.items():
    x = ol[m].dropna(subset=['d_f', 'fcstvar'])
    R = x.d_f.var() - x.fcstvar.mean()
    rows.append({'screen': name, 'n': len(x), 'kept_%': 100 * len(x) / ol.coherency_ratio.notna().sum(),
                 'bias': x.d_f.mean(), 'R': R, 'errstd': np.sqrt(max(R, 0))})
display(pd.DataFrame(rows).round(3))

## 7. Vegetation and land cover

The operator applies **no dynamic vegetation correction** — opacity is frozen at
build-time climatology — so error it cannot model should appear where vegetation is
densest.

In [ ]:
veg = implied_R(ol, 'coefficient_weighted_opacity')
display(veg[['n', 'lo', 'hi', 'bias', 'fcstvar', 'R', 'errstd']].round(3))
lc = implied_R(ol, 'modis_land_cover', categorical=True).sort_values('R')
display(lc[['n', 'bias', 'fcstvar', 'R', 'errstd']].round(3))

fig, ax = plt.subplots(1, 2, figsize=(11.5, 3.6))
x = np.arange(len(veg))
ax[0].bar(x, veg.errstd, 0.6, color='#54a24b')
ax[0].set_xticks(x)
ax[0].set_xticklabels([f'{a:.3f}-{b:.3f}' for a, b in zip(veg.lo, veg.hi)], fontsize=7.5, rotation=20)
ax[0].set_xlabel('vegetation optical depth'); ax[0].set_ylabel('implied errstd (dB)')
ax[0].set_title('(a) vegetation attenuation', loc='left', fontweight='bold')
xl = np.arange(len(lc))
ax[1].bar(xl, lc.errstd, 0.6, color='#f58518')
ax[1].set_xticks(xl); ax[1].set_xticklabels([int(i) for i in lc.index])
ax[1].set_xlabel('MODIS land-cover class'); ax[1].set_ylabel('implied errstd (dB)')
ax[1].set_title('(b) by land cover  (12 crop, 7 shrub, 8 savanna)', loc='left', fontweight='bold')
fig.tight_layout()

## 8. Are coherency and vegetation the same observations?

If they flag the same ones, a single screen suffices. If not, they compound.

In [ ]:
s = ol.dropna(subset=['coherency_ratio', 'coefficient_weighted_opacity', 'd_f', 'fcstvar']).copy()
VEG_T = s.coefficient_weighted_opacity.quantile(0.80)
s['low_coh'] = s.coherency_ratio < P20
s['high_veg'] = s.coefficient_weighted_opacity > VEG_T
print(f'thresholds: coherency < {P20:.3f}, opacity > {VEG_T:.3f}')

rows = []
for lc_ in (False, True):
    for hv in (False, True):
        x = s[(s.low_coh == lc_) & (s.high_veg == hv)]
        R = x.d_f.var() - x.fcstvar.mean()
        rows.append({'coherency': 'LOW' if lc_ else 'ok', 'vegetation': 'HIGH' if hv else 'ok',
                     'n': len(x), 'pct': 100 * len(x) / len(s),
                     'bias': x.d_f.mean(), 'R': R, 'errstd': np.sqrt(max(R, 0))})
display(pd.DataFrame(rows).round(2))

print(f'flagged by both : {100*(s.low_coh & s.high_veg).mean():.1f}%')
print(f'if independent  : {100*s.low_coh.mean()*s.high_veg.mean():.1f}%')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2),
                         subplot_kw={'projection': ccrs.PlateCarree()})
for ax, (col, title) in zip(axes, [('low_coh', '(a) fraction failing coherency screen'),
                                   ('high_veg', '(b) fraction failing vegetation screen')]):
    lo, la, gr = binned(s, s[col].astype(float), minc=20)
    basemap(ax)
    m = ax.pcolormesh(lo, la, gr, cmap='OrRd', vmin=0, vmax=1,
                      transform=ccrs.PlateCarree(), zorder=1)
    fig.colorbar(m, ax=ax, fraction=0.04, label='fraction screened out')
    ax.set_title(title, loc='left', fontweight='bold')
fig.tight_layout()

In [ ]:
# Report figure: why coherency was chosen as the screen variable.
REPORT_OUT = PROJECT / 'output' / 'coherency_screening_figures'
REPORT_OUT.mkdir(parents=True, exist_ok=True)

coh_ol = implied_R(ol, 'coherency_ratio')
coh_da = implied_R(da, 'coherency_ratio')
x = np.arange(len(coh_ol))
labels = [f'{a:.2f}-{b:.2f}' for a, b in zip(coh_ol.lo, coh_ol.hi)]

fig, ax = plt.subplots(1, 2, figsize=(9.2, 3.6))
ax[0].bar(x - 0.18, coh_ol.errstd, 0.36, color='#4c78a8', label='open loop')
ax[0].bar(x + 0.18, coh_da.errstd.reindex(coh_ol.index), 0.36, color='#9ecae9', label='assimilating')
ax[0].axhline(4.4, color='#e45756', ls='--', lw=1.1)
ax[0].text(0.03, 4.5, 'configured 4.4 dB', color='#e45756', fontsize=8)
ax[0].set_ylabel('implied CYGNSS L1 errstd (dB)')
ax[0].set_title('(a) lowest coherency bin has ~1.9x errstd', loc='left', fontweight='bold')
ax[0].legend(frameon=False, fontsize=8)

colors = ['#e45756' if v < 0 else '#54a24b' for v in coh_ol.bias]
ax[1].bar(x, coh_ol.bias, 0.58, color=colors)
ax[1].axhline(0, color='0.25', lw=0.9)
ax[1].set_ylabel('mean innovation O$-$F (dB)')
ax[1].set_title('(b) coherency also carries bias structure', loc='left', fontweight='bold')

for a in ax:
    a.set_xticks(x)
    a.set_xticklabels(labels, rotation=25, ha='right', fontsize=8)
    a.set_xlabel('coherency-ratio quintile')
fig.suptitle('CYGNSS L1 coherency identifies a high-error-variance observation subset', y=1.04)
fig.tight_layout()
path = REPORT_OUT / 'report_coherency_implied_error_and_bias.png'
fig.savefig(path, dpi=220, bbox_inches='tight')
path.relative_to(PROJECT)


## 9. Summary

**Analysis response** *(assimilating run)* — mechanically the filter works. 73% of
updates move the right way and the response scales with the gain. It also shows
*where* performance is weaker, which is a result in its own right.

**Observation error** *(open loop, 3 years, no feedback)* — implied $\hat{R}$ gives a
gain-independent view, and the two runs agree on the ordering:

| screen | rationale |
|---|---|
| `coherency_ratio >= P20` | bottom quintile carries ~3× the error variance; keeps ~80% |
| `coefficient_weighted_opacity <= P80` | top quintile roughly doubles it; operator has no dynamic vegetation correction |

**One-sided, not two-sided.** Cutting the top coherency quintile as well buys little
error reduction, costs 20% of the data, and leaves the retained population *more*
biased. The two screens are near-independent, so they compound.

**What this does not establish.** That screening improves the analysis — that needs a
run with the screen applied, compared against a **randomly thinned run at matched
observation count**, since removing any observations helps if assimilation is
currently harmful. And $\hat{R}$ does not separate instrument noise from operator
error, so the vegetation result says the *system* does worse there, not that CYGNSS
is noisier.

**Left open:** the coherency-dependent bias is invisible to $\hat{R}$ and is not
removed by the per-tile, per-pentad scaling.